# Chapter 16 Natural Language Processing with RNNs and Attention
## 16.1 Generating Shakespearean Text Using a Character RNN
### 16.1.7 Stateful RNN

#### Dataset

In [1]:
import tensorflow as tf
from typing import Union
from yarl import URL


def load_text(url: Union[str, URL]) -> str:
    with open(tf.keras.utils.get_file(fname="tmp.txt", origin=str(url))) as fp:
        return fp.read()


data_text = load_text(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
)
data_text[:100]

2023-03-18 09:47:35.596127: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

#### Tokenizer and Encoded Data

In [2]:
tokenizer = tf.keras.preprocessing.text.Tokenizer(char_level=True)
tokenizer.fit_on_texts(data_text)
tokenizer

In [3]:
n_chars = len(tokenizer.word_counts)
n_chars

39

In [4]:
import numpy as np
import numpy.typing as npt


# def encode_text(text: str) -> npt.NDArray[int]:
def encode_text(text: str) -> npt.NDArray[int]:
    return np.array(tokenizer.texts_to_sequences([data_text])) - 1


[data_encoded] = encode_text(data_text)
print("shape:", data_encoded.shape)
data_encoded[:10]

shape: (1115394,)


array([19,  5,  8,  7,  2,  0, 18,  5,  2,  5])

#### Preprocessing Dataset

##### Train-Test Split

In [5]:
size_dataset = tokenizer.document_count
size_dataset

1115394

In [6]:
size_train = round(size_dataset * 0.9)
size_valid = size_dataset - size_train
size_test = 0
size_train

1003855

In [7]:
text_train = data_encoded[:size_train]
text_valid = data_encoded[size_train:-size_test]

In [8]:
batch_size = 32
n_steps = 100
shift = 1
window_length = n_steps + shift

text_parts_train = np.array_split(text_train, batch_size)

datasets = []
for part in text_parts_train:
    dataset = tf.data.Dataset.from_tensor_slices(part)
    dataset = dataset.window(size=window_length, shift=n_steps, drop_remainder=True)
    dataset = dataset.flat_map(lambda window: window.batch(window_length))
    datasets.append(dataset)

dataset_train_zip = tf.data.Dataset.zip(tuple(datasets)).map(
    lambda *entry: tf.stack(entry)
)


def print_dataset_head(dataset: tf.data.Dataset, n_instances: int) -> npt.NDArray:
    for ins in dataset.take(n_instances):
        print(ins)
        if isinstance(ins, tf.data.Dataset):
            print(list(ins.as_numpy_iterator()))


print_dataset_head(dataset_train_zip, 3)

2023-03-18 09:47:38.951378: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-03-18 09:47:38.983704: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-03-18 09:47:38.984104: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-03-18 09:47:38.984856: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropri

tf.Tensor(
[[19  5  8 ...  3 13  0]
 [ 7  1 31 ...  6  1  0]
 [18  5  9 ... 11 12  0]
 ...
 [ 2  0  5 ...  9  7  1]
 [25  3  7 ... 25  1  8]
 [ 1  7  0 ...  2  3 18]], shape=(32, 101), dtype=int64)
tf.Tensor(
[[ 0  4  8 ...  3 13  0]
 [ 0 16  4 ... 31 31 10]
 [ 0 15  3 ...  0 16  4]
 ...
 [ 1 33 13 ...  3  2  0]
 [ 8  0  2 ... 20  3  0]
 [18 24  5 ...  4  7  0]], shape=(32, 101), dtype=int64)
tf.Tensor(
[[ 0 24  9 ... 13  7  0]
 [10 16  6 ... 11 28 10]
 [ 4  7  0 ...  2  1  8]
 ...
 [ 0  4 14 ...  0  4  0]
 [ 0 18  4 ...  9 12  0]
 [ 0 14 15 ...  4  2  0]], shape=(32, 101), dtype=int64)


In [9]:
dataset_train_split = dataset_train_zip.map(
    lambda window: (tf.one_hot(window[:, :-1], depth=n_chars), window[:, 1:])
)
print_dataset_head(dataset_train_split, 3)

(<tf.Tensor: shape=(32, 100, 39), dtype=float32, numpy=
array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       ...,

       [[0., 0., 1., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0.

In [10]:
dataset_train_prep = dataset_train_split.prefetch(1)

In [11]:
def text2dataset(text, batch, n_steps, window_length) -> tf.data.Dataset:
    text_parts = np.array_split(text, batch)
    datasets = []
    for part in text_parts:
        dataset = tf.data.Dataset.from_tensor_slices(part)
        dataset = dataset.window(size=window_length, shift=n_steps, drop_remainder=True)
        dataset = dataset.flat_map(lambda window: window.batch(window_length))
        datasets.append(dataset)
    dataset = tf.data.Dataset.zip(tuple(datasets)).map(lambda *entry: tf.stack(entry))
    dataset = dataset.map(
        lambda window: (tf.one_hot(window[:, :-1], depth=n_chars), window[:, 1:])
    )
    return dataset.prefetch(1)


dataset_valid_prep = text2dataset(
    text_valid, batch=batch_size, n_steps=n_steps, window_length=window_length
)

#### Model

In [12]:
import tensorflow as tf

model = tf.keras.models.Sequential([
    tf.keras.layers.GRU(128, return_sequences=True, stateful=True, batch_input_shape=[batch_size, None, n_chars]),
    tf.keras.layers.GRU(128, return_sequences=True, stateful=True),
    tf.keras.layers.Dense(n_chars, activation="softmax"),
])

class ResetStateCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs):
        self.model.reset_states()

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics="acc")
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 gru (GRU)                   (32, None, 128)           64896     
                                                                 
 gru_1 (GRU)                 (32, None, 128)           99072     
                                                                 
 dense (Dense)               (32, None, 39)            5031      
                                                                 
Total params: 168,999
Trainable params: 168,999
Non-trainable params: 0
_________________________________________________________________


In [13]:
import time
from pathlib import Path

root_logdir = Path().absolute() / "logs"

%load_ext tensorboard
%tensorboard --logdir=./logs --port=6006

Launching TensorBoard...

In [14]:
early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
tensorboard_cb = tf.keras.callbacks.TensorBoard(root_logdir / time.strftime("run_%Y_%m_%d-%H_%M_%S"))

history = model.fit(dataset_train_prep, validation_data=dataset_valid_prep, epochs=50, callbacks=[early_stopping_cb, tensorboard_cb, ResetStateCallback()])
# history = model.fit(dataset_train_prep, epochs=50, callbacks=[ResetStateCallback()])

Epoch 1/50


2023-03-18 09:47:46.363169: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8401


      5/Unknown - 3s 15ms/step - loss: 3.6217 - acc: 0.1011WARNING:tensorflow:Callback method `on_train_batch_end` is slow compared to the batch time (batch time: 0.0091s vs `on_train_batch_end` time: 0.0114s). Check your callbacks.
    312/Unknown - 7s 16ms/step - loss: 2.4838 - acc: 0.2965WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,acc
313/313 [==============================] - 9s 20ms/step - loss: 2.4827 - acc: 0.2968
Epoch 2/50
313/313 [==============================] - 7s 21ms/step - loss: 1.9833 - acc: 0.4135
Epoch 3/50
313/313 [==============================] - 7s 22ms/step - loss: 1.7805 - acc: 0.4670
Epoch 4/50
313/313 [==============================] - 8s 24ms/step - loss: 1.6636 - acc: 0.4988
Epoch 5/50
313/313 [==============================] - 7s 24ms/step - loss: 1.5900 - acc: 0.5187
Epoch 6/50
313/313 [==============================] - 8s 24ms/step - loss: 1.5384 - acc: 0.5326
Epoch 7/50
313/313 [

In [15]:
dir_models = Path() / "models"
model.save(dir_models / "stateful_rnn")

INFO:tensorflow:Assets written to: models/stateful_rnn/assets


INFO:tensorflow:Assets written to: models/stateful_rnn/assets


#### Usage

##### Model

In [28]:
# Model
from pathlib import Path
import tensorflow as tf

path_model = Path() / "models" / "stateful_rnn"
model_stateful = tf.keras.models.load_model(path_model)

model_stateless = tf.keras.models.Sequential([
    tf.keras.layers.GRU(128, return_sequences=True, batch_input_shape=[None, None, n_chars]),
    tf.keras.layers.GRU(128, return_sequences=True),
    tf.keras.layers.Dense(n_chars, activation="softmax"),
])

model_stateless.set_weights(model_stateful.get_weights())

In [32]:
# Process
from typing import Sequence

def preprocess(texts: Sequence[str]) -> tf.Tensor:
    return tf.one_hot(np.array(tokenizer.texts_to_sequences(texts)) - 1, depth=n_chars)

def predict_next(text: str, temperature: float = 1.0) -> str:
    y_proba = model_stateless(preprocess([text]))[0, -1:]
    logits = tf.math.log(y_proba) / temperature
    id_char = tf.random.categorical(logits, num_samples=1) + 1
    return tokenizer.sequences_to_texts(id_char.numpy())[0]

def predict_text(text: str, n_chars: int = 50, temperature: int = 1) -> str:
    for _ in range(n_chars):
        text += predict_next(text)
    return text

In [33]:
text = predict_text("Do yo")
print(text)

Do yours,
that widwill so take no fruits my store.


pa
